# Newsvendor Simulation
A reproducible Python replacement for the Excel newsvendor "tools":
- Single-item newsvendor (critical fractile, z, Q*)
- Multi-item newsvendor with capacity limit (shadow price theta)
- Monte Carlo simulation of profit vs Q and profit vs capacity
- Sensitivity analysis (sigma, salvage, shipping / free shipping)

## 1. Imports & Configuration

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# Global simulation defaults
N_SIMS: int = 100_000
SEED: int = 7

print("Imports OK. numpy:", np.__version__)


## 2. Core Economics Functions

Define `effective_unit_cost`, `underage_cost`, `overage_cost`, `critical_fractile`, and `newsvendor_q_normal`.

In [ ]:
def effective_unit_cost(production_cost: float, shipping_cost: float, free_shipping: bool) -> float:
    """If free_shipping=True, you pay shipping per unit produced."""
    return production_cost + (shipping_cost if free_shipping else 0.0)

def underage_cost(price: float, unit_cost: float) -> float:
    """Lost profit when demand exceeds supply."""
    return price - unit_cost

def overage_cost(unit_cost: float, salvage: float) -> float:
    """Waste cost when supply exceeds demand."""
    return unit_cost - salvage

def critical_fractile(u: float, o: float) -> float:
    """CF = u / (u + o). Optimal service level target."""
    if u <= 0 or o < 0:
        raise ValueError(f"Need u>0 and o>=0. Got u={u}, o={o}")
    return u / (u + o)

def newsvendor_q_normal(mu: float, sigma: float, u: float, o: float) -> float:
    """Q* = mu + z * sigma, where z = norm.ppf(CF)."""
    cf = critical_fractile(u, o)
    z = norm.ppf(cf)
    return mu + z * sigma

# --- Worked example ---
price      = 100.0
prod_cost  = 30.0
ship_cost  = 10.0
salvage    = 0.0
mu         = 200.0
sigma      = 60.0

unit_cost  = effective_unit_cost(prod_cost, ship_cost, free_shipping=True)
u          = underage_cost(price, unit_cost)
o          = overage_cost(unit_cost, salvage)
cf         = critical_fractile(u, o)
q_star     = newsvendor_q_normal(mu, sigma, u, o)

print("=== Worked Example ===")
print(f"  unit_cost  : {unit_cost:.2f}")
print(f"  u (under)  : {u:.2f}")
print(f"  o (over)   : {o:.2f}")
print(f"  CF         : {cf:.4f}")
print(f"  z          : {norm.ppf(cf):.4f}")
print(f"  Q*         : {q_star:.2f}")


## 3. Demand Sampling

Sample normal demand truncated at zero and verify visually.

In [ ]:
def sample_demand_normal(mu: float, sigma: float, n: int, rng: np.random.Generator) -> np.ndarray:
    """Normal demand truncated at 0 (no negative demand)."""
    d = rng.normal(loc=mu, scale=sigma, size=n)
    return np.maximum(d, 0.0)

# Plot histogram of 10 000 demand samples
rng_demo = np.random.default_rng(SEED)
demand_samples = sample_demand_normal(mu, sigma, 10_000, rng_demo)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(demand_samples, bins=50, color="steelblue", edgecolor="white", density=True)
x_range = np.linspace(demand_samples.min(), demand_samples.max(), 200)
ax.plot(x_range, norm.pdf(x_range, mu, sigma), "r--", linewidth=2, label="Normal PDF")
ax.axvline(mu, color="orange", linestyle="--", label=f"μ = {mu:.0f}")
ax.set_xlabel("Demand")
ax.set_ylabel("Density")
ax.set_title(f"Demand Distribution  (μ={mu}, σ={sigma})")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Sample mean: {demand_samples.mean():.2f}  |  Sample std: {demand_samples.std():.2f}")


## 4. Single-Item Profit Simulation

Vectorised profit function + Monte Carlo expected profit at Q*.

In [ ]:
def profit_single_item(
    demand: np.ndarray,
    Q: float,
    price: float,
    unit_cost: float,
    salvage: float,
) -> np.ndarray:
    """
    Profit per scenario:
      sold     = min(demand, Q)
      leftover = max(Q - demand, 0)
      profit   = price*sold - unit_cost*Q + salvage*leftover
    """
    sold     = np.minimum(demand, Q)
    leftover = np.maximum(Q - demand, 0.0)
    return price * sold - unit_cost * Q + salvage * leftover


def expected_profit_single_item(
    mu: float,
    sigma: float,
    Q: float,
    price: float,
    unit_cost: float,
    salvage: float,
    n_sims: int = N_SIMS,
    seed: int = SEED,
) -> Dict[str, float]:
    rng  = np.random.default_rng(seed)
    d    = sample_demand_normal(mu, sigma, n_sims, rng)
    prof = profit_single_item(d, Q, price, unit_cost, salvage)
    return {
        "Q":           float(Q),
        "mean_profit": float(np.mean(prof)),
        "p10_profit":  float(np.percentile(prof, 10)),
        "p50_profit":  float(np.percentile(prof, 50)),
        "p90_profit":  float(np.percentile(prof, 90)),
        "mean_demand": float(np.mean(d)),
    }


# --- Print result for Q* ---
result = expected_profit_single_item(mu, sigma, q_star, price, unit_cost, salvage)
print("=== MC Profit at Q* ===")
for k, v in result.items():
    print(f"  {k:<15}: {v:.2f}")


## 5. Multi-Item Dataclass & Shadow Price (Theta) Solver

Bisection solver replaces Excel Goal Seek.

In [ ]:
@dataclass(frozen=True)
class Item:
    name:  str
    mu:    float
    sigma: float
    u:     float
    o:     float


def cf_with_theta(u: float, o: float, theta: float, eps: float = 1e-9) -> float:
    """
    CF_i(theta) = (u_i - theta) / (u_i + o_i)
    Clamped to (eps, 1-eps) to avoid norm.ppf blowing up.
    """
    cf = (u - theta) / (u + o)
    return min(max(cf, eps), 1 - eps)


def q_with_theta(item: Item, theta: float) -> float:
    cf = cf_with_theta(item.u, item.o, theta)
    z  = norm.ppf(cf)
    return item.mu + z * item.sigma


def solve_theta_for_capacity(
    items: List[Item],
    capacity: float,
    theta_low: float = 0.0,
    theta_high: float = 1_000.0,
    tol: float = 1e-6,
    max_iter: int = 200,
) -> float:
    """Bisection: find theta s.t. sum Q_i(theta) == capacity."""

    def total_q(theta: float) -> float:
        return float(sum(q_with_theta(it, theta) for it in items))

    lo, hi = theta_low, theta_high

    if total_q(lo) <= capacity:
        return lo

    while total_q(hi) > capacity:
        hi *= 2.0
        if hi > 1e12:
            raise RuntimeError("Could not bracket theta. Check capacity vs item parameters.")

    for _ in range(max_iter):
        mid = (lo + hi) / 2.0
        tq  = total_q(mid)
        if abs(tq - capacity) <= tol:
            return mid
        if tq > capacity:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0


def allocate_with_capacity(items: List[Item], capacity: float) -> Tuple[float, List[Tuple[str, float]]]:
    theta = solve_theta_for_capacity(items, capacity)
    alloc = [(it.name, float(q_with_theta(it, theta))) for it in items]
    return float(theta), alloc


# --- Example: two-item allocation at capacity=400 ---
items_demo = [
    Item(name="A", mu=100.0, sigma=10.0, u=4.0, o=1.0),
    Item(name="B", mu=300.0, sigma=10.0, u=4.0, o=1.0),
]
theta_400, alloc_400 = allocate_with_capacity(items_demo, capacity=400.0)
print(f"Capacity = 400  |  theta = {theta_400:.4f}")
for name, q in alloc_400:
    print(f"  Q_{name} = {q:.2f}")
print(f"  Total Q = {sum(q for _, q in alloc_400):.2f}")


## 6. Capacity Sweep

Iterate over a range of capacities and collect theta + per-item Q.

In [ ]:
def capacity_sweep(items: List[Item], capacities: List[float]) -> List[Dict[str, float]]:
    rows: List[Dict[str, float]] = []
    for cap in capacities:
        theta, alloc = allocate_with_capacity(items, cap)
        row: Dict[str, float] = {"capacity": float(cap), "theta": float(theta)}
        for name, q in alloc:
            row[f"Q_{name}"] = float(q)
        row["total_Q"] = float(sum(q for _, q in alloc))
        rows.append(row)
    return rows


capacities = list(np.linspace(360, 460, 11))
sweep_rows = capacity_sweep(items_demo, capacities)

# Display as a formatted table
header = f"{'Capacity':>10} {'theta':>8} {'Q_A':>8} {'Q_B':>8} {'Total':>8}"
print(header)
print("-" * len(header))
for r in sweep_rows:
    print(
        f"{r['capacity']:>10.0f} {r['theta']:>8.4f} "
        f"{r['Q_A']:>8.2f} {r['Q_B']:>8.2f} {r['total_Q']:>8.2f}"
    )


## 7. Plotting Utilities

Reusable plot functions for profit curves and capacity allocation.

In [ ]:
def plot_profit_curve(
    q_grid: np.ndarray,
    profits: np.ndarray,
    title: str,
    q_star: Optional[float] = None,
) -> None:
    """Plot expected profit vs Q with an optional vertical line at Q*."""
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(q_grid, profits, color="steelblue", linewidth=2)
    if q_star is not None:
        ax.axvline(q_star, color="red", linestyle="--", linewidth=1.5, label=f"Q* = {q_star:.1f}")
        ax.legend()
    ax.set_xlabel("Q (Production Quantity)")
    ax.set_ylabel("Expected Profit (Monte Carlo)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_capacity_alloc(rows: List[Dict[str, float]], item_names: List[str]) -> None:
    """Plot Q per item and theta vs capacity."""
    caps = [r["capacity"] for r in rows]
    fig, ax1 = plt.subplots(figsize=(9, 4))

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for i, nm in enumerate(item_names):
        ax1.plot(caps, [r[f"Q_{nm}"] for r in rows], marker="o", label=f"Q_{nm}", color=colors[i])

    ax1.set_xlabel("Capacity")
    ax1.set_ylabel("Optimal Quantity")
    ax1.set_title("Optimal Allocation & Shadow Price vs Capacity")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(caps, [r["theta"] for r in rows], "k--", marker="s", linewidth=1.5, label="theta (θ)")
    ax2.set_ylabel("Shadow Price (θ)")

    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.show()


print("Plotting utilities defined.")


## 8. Demo: Single-Item Profit Curve

Monte Carlo expected profit over a Q grid centred on Q*.

In [ ]:
# Build Q grid: Q* ± 3σ with 61 points
q_grid = np.linspace(max(0.0, q_star - 3 * sigma), q_star + 3 * sigma, 61)

mean_profits = []
for q in q_grid:
    stats = expected_profit_single_item(
        mu, sigma, q, price, unit_cost, salvage, n_sims=80_000, seed=11
    )
    mean_profits.append(stats["mean_profit"])

print(f"Q* = {q_star:.2f}  |  Max MC profit ≈ {max(mean_profits):.2f}  at Q ≈ {q_grid[int(np.argmax(mean_profits))]:.2f}")

plot_profit_curve(
    q_grid,
    np.array(mean_profits),
    "Expected Profit vs Q  (Single Item, Monte Carlo)",
    q_star=q_star,
)


## 9. Demo: Multi-Item Capacity Sweep

Show how Q_A, Q_B, and shadow price θ respond as capacity varies from 360 → 460.

In [ ]:
capacities_sweep = list(np.linspace(360, 460, 11))
sweep_results = capacity_sweep(items_demo, capacities_sweep)

plot_capacity_alloc(sweep_results, item_names=[it.name for it in items_demo])

print("\nKey observations:")
print("  • As capacity increases, theta (shadow price) decreases toward 0.")
print("  • Both Q_A and Q_B grow as more capacity becomes available.")
print("  • When capacity is non-binding, Q_A and Q_B converge to their standalone Q* values.")
standalone_a = newsvendor_q_normal(items_demo[0].mu, items_demo[0].sigma, items_demo[0].u, items_demo[0].o)
standalone_b = newsvendor_q_normal(items_demo[1].mu, items_demo[1].sigma, items_demo[1].u, items_demo[1].o)
print(f"  Standalone Q*_A = {standalone_a:.2f}  |  Q*_B = {standalone_b:.2f}")
